## Notebook for identifying TF communities in the Norman dataset

In [1]:
# set up
import numpy as np
import pandas as pd
import math
from scipy.stats import false_discovery_control
import pickle
import gseapy as gp

# Import GRN analysis functions from gcrl package
from gcrl.grn import (
    build_tf_tf_regulatory_layer,
    build_tf_tf_cotarget_layer,
    sweep_hyperparams,
    summarize_all_stabilities,
    build_coassociation_matrix,
    consensus_partition_from_coassoc,
    community_stats,
    run_single_layer_leiden_reg,
    run_single_layer_leiden_cot,
    run_ora_for_clusters,
    filter_gene_sets_by_size_and_level,
    build_cluster_gene_sets,
    extract_go_ids_from_terms,
    compute_go_levels
)

In [12]:
# control parameters
top_percentage = 0.50  # percentage of top edges to keep in the final GRN
data_folder = '../../../data/real/Norman2019/'  # path to data folder
significance_level = 0.05  # significance level for FDR correction
min_size = 5  # minimum cluster size for merging small clusters

### Loading and filtering GRN

In [3]:
#Load GRN 
raw_grn = pd.read_csv(data_folder + 'GRN/' + "raw_GRN.csv", index_col=0)

#Drop NA values in p-value column
raw_grn = raw_grn.dropna(subset=["p"])

#FDR filtering and only leaving adj p-values values less than 0.05
raw_grn.loc[:, 'adjp'] = false_discovery_control(raw_grn['p'].values, method='bh')
filtered_grn = raw_grn[raw_grn['adjp'] <= significance_level].copy()

# Sort by absolute coefficient descending
filtered_grn = filtered_grn.sort_values(by='coef_abs', ascending=False).reset_index(drop=True)

# selecting top_percentage of top edges
num_top_edges = math.ceil(len(filtered_grn) * top_percentage)
current_grn = filtered_grn.iloc[:num_top_edges, :]

# print statistics
print(current_grn['source'].nunique(), current_grn['target'].nunique())
current_grn


141 2048


,source,target,coef_mean,coef_abs,p,-logp,adjp
0,JUND,HIST1H4C,0.801887,0.801887,6.427243e-13,12.191975,6.133731e-12
1,PITX1,HIST1H4C,0.566257,0.566257,7.473443e-08,7.126479,1.766064e-07
2,KLF1,HBZ,0.548236,0.548236,8.472660e-20,19.071980,2.997454e-17
3,LYL1,HSP90AA1,0.462909,0.462909,1.499360e-25,24.824094,1.036404e-21
4,ENO1,RAN,0.459944,0.459944,7.764523e-19,18.109885,1.465798e-16
...,...,...,...,...,...,...,...
38349,PRDM1,MVP,-0.000333,0.000333,5.406043e-14,13.267121,7.915708e-13
38350,FOS,LINC00987,-0.000333,0.000333,9.313351e-06,5.030894,1.597743e-05
38351,MEF2C,SHC4,-0.000333,0.000333,1.662021e-10,9.779363,7.130883e-10
38352,KLF2,SLC16A3,0.000333,0.000333,1.868734e-08,7.728453,4.961866e-08


### Retrieving the list of TFs

In [4]:
# Load TF reference
tf_info = pd.read_parquet(data_folder + 'GRN/' + "hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet") ## Google Drive: gCAL_data > Norman_results
tf_names = list(tf_info.columns.unique())
tf_names = [item for item in tf_names if item in current_grn['source'].unique() or item in current_grn['target'].unique()]
print(f"Number of TFs in reference: {len(tf_names)}")

Number of TFs in reference: 141


## Initial assessments

In [8]:
# 1. Build layers
g_reg, tf_index = build_tf_tf_regulatory_layer(current_grn, tf_names)
g_cot, sim_matrix = build_tf_tf_cotarget_layer(current_grn, tf_names, tf_index, min_similarity=0.15)
print('Number of edges in g_reg: ', g_reg.ecount())
print('Number of edges in g_cot: ', g_cot.ecount())

# 2. computing memberships
mem_reg, _, _ = run_single_layer_leiden_reg(g_reg, gamma_reg=1.2, seed=0)
mem_cot, _, _ = run_single_layer_leiden_cot(g_cot, gamma_cot=1.2, seed=0)

# 3. computing community statistics
print(community_stats(g_reg, mem_reg).head(n=10))
print(community_stats(g_reg, mem_reg).shape)
print(community_stats(g_cot, mem_cot).head(n=10))
print(community_stats(g_cot, mem_cot).shape)


Number of edges in g_reg:  2798
Number of edges in g_cot:  2787
   community  size  w_internal   density
0          0    24    1.412644  0.005118
3          1    23    0.955801  0.003778
1          2    19    0.584122  0.003416
2          3    18    1.073140  0.007014
4          4    15    0.354603  0.003377
6          5    15    0.790294  0.007527
7          6    13    0.701937  0.008999
5          7    12    0.259542  0.003932
8          8     1    0.000000  0.000000
9          9     1    0.000000  0.000000
(10, 4)
   community  size  w_internal   density
1          0    31   31.062353  0.066801
4          1    23   53.919858  0.213122
5          2    21   21.858653  0.104089
6          3    18   60.977836  0.398548
3          4    17   27.209557  0.200070
0          6    15   18.569932  0.176856
2          5    15   20.014301  0.190612
7          7     1    0.000000  0.000000
(8, 4)


In [9]:

# 2. Hyperparameter grids (tune as needed)
gamma_reg_list = [1.2, 1.25, 1.3]
gamma_cot_list = [1.2, 1.25, 1.3]
cot_weight_list = [0.1, 0.5, 1.0]   # plays the role of "ω"

# 3. Sweep & collect partitions
results = sweep_hyperparams(
    g_reg,
    g_cot,
    gamma_reg_list=gamma_reg_list,
    gamma_cot_list=gamma_cot_list,
    cot_weight_list=cot_weight_list,
    n_seeds=100,
    n_iterations=-1,   # until convergence
)

Done γ_reg=1.2, γ_cot=1.2, w_cot=0.1
Done γ_reg=1.2, γ_cot=1.2, w_cot=0.5
Done γ_reg=1.2, γ_cot=1.2, w_cot=1.0
Done γ_reg=1.2, γ_cot=1.25, w_cot=0.1
Done γ_reg=1.2, γ_cot=1.25, w_cot=0.5
Done γ_reg=1.2, γ_cot=1.25, w_cot=1.0
Done γ_reg=1.2, γ_cot=1.3, w_cot=0.1
Done γ_reg=1.2, γ_cot=1.3, w_cot=0.5
Done γ_reg=1.2, γ_cot=1.3, w_cot=1.0
Done γ_reg=1.25, γ_cot=1.2, w_cot=0.1
Done γ_reg=1.25, γ_cot=1.2, w_cot=0.5
Done γ_reg=1.25, γ_cot=1.2, w_cot=1.0
Done γ_reg=1.25, γ_cot=1.25, w_cot=0.1
Done γ_reg=1.25, γ_cot=1.25, w_cot=0.5
Done γ_reg=1.25, γ_cot=1.25, w_cot=1.0
Done γ_reg=1.25, γ_cot=1.3, w_cot=0.1
Done γ_reg=1.25, γ_cot=1.3, w_cot=0.5
Done γ_reg=1.25, γ_cot=1.3, w_cot=1.0
Done γ_reg=1.3, γ_cot=1.2, w_cot=0.1
Done γ_reg=1.3, γ_cot=1.2, w_cot=0.5
Done γ_reg=1.3, γ_cot=1.2, w_cot=1.0
Done γ_reg=1.3, γ_cot=1.25, w_cot=0.1
Done γ_reg=1.3, γ_cot=1.25, w_cot=0.5
Done γ_reg=1.3, γ_cot=1.25, w_cot=1.0
Done γ_reg=1.3, γ_cot=1.3, w_cot=0.1
Done γ_reg=1.3, γ_cot=1.3, w_cot=0.5
Done γ_reg=1.3, γ_co

In [10]:

# 4. Stability summaries
stability = summarize_all_stabilities(
    results,
    compute_ari=True,   # no sklearn, no warnings, faster
    max_pairs=None        # or None if you want all pairs
)
#stability  # dict: (gamma_reg, gamma_cot, omega) -> ARI/VI summary

In [ ]:
# saving / loading
#with open("multiplex_results.pkl", "wb") as f:
#    pickle.dump(results, f)
#with open("multiplex_stability.pkl", "wb") as f:
#    pickle.dump(stability, f)
#with open("multiplex_results.pkl", "rb") as f:
#    results = pickle.load(f)
#with open("multiplex_stability.pkl", "rb") as f:
#    stability = pickle.load(f)


In [23]:
# Pick the most stable triple
best_key = max(stability.items(), key=lambda kv: kv[1]['ARI_mean'])[0]
print("Best hyperparams:", best_key)
print("Best stability stats:", stability[best_key])

# 5. Consensus clustering for best hyperparams
memberships_list = results[best_key]
C = build_coassociation_matrix(memberships_list)
consensus_labels, consensus_graph = consensus_partition_from_coassoc(
    C, min_coassoc=0.5, resolution=1.5
)

# 6. Diagnostics per layer
df_reg_stats = community_stats(g_reg, consensus_labels)
df_cot_stats = community_stats(g_cot, consensus_labels)

print("Regulatory layer community stats:")
print(df_reg_stats)

print("Co-target layer community stats:")
print(df_cot_stats)


Best hyperparams: (1.2, 1.3, 0.5)
Best stability stats: {'VI_mean': np.float64(1.1663778380249639), 'VI_std': np.float64(0.29032647168182346), 'ARI_mean': np.float64(0.6150388303246095), 'ARI_std': np.float64(0.09752662972367575), 'n_pairs': 4950}
Regulatory layer community stats:
    community  size  w_internal   density
1           0    24    0.274469  0.000994
2           1    22    0.080087  0.000347
4           2    21    1.448115  0.006896
6           3    17    0.443530  0.003261
0           4    14    0.435412  0.004785
10          5    12    0.334627  0.005070
11          6    10    0.010824  0.000241
8           7     5    0.006688  0.000669
5           8     4    0.002044  0.000341
7           9     4    0.002988  0.000498
14         10     3    0.006008  0.002003
3          11     1    0.000000  0.000000
9          12     1    0.000000  0.000000
12         13     1    0.000000  0.000000
13         14     1    0.000000  0.000000
15         15     1    0.000000  0.000000
Co-t

In [24]:
# Count elements per cluster
counts = pd.DataFrame({"consensus_labels": consensus_labels})["consensus_labels"].value_counts().to_dict()

# Identify small clusters
small_clusters = {lab for lab, cnt in counts.items() if cnt < min_size}

# Replace small clusters with the minimum label
min_label = int(np.min(np.array(list(small_clusters))))
consensus_labels = np.where(np.isin(consensus_labels, list(small_clusters)), min_label, consensus_labels)

# 6. Diagnostics per layer
df_reg_stats = community_stats(g_reg, consensus_labels)
df_cot_stats = community_stats(g_cot, consensus_labels)

print("Regulatory layer community stats:")
print(df_reg_stats)

print("Co-target layer community stats:")
print(df_cot_stats)


Regulatory layer community stats:
   community  size  w_internal   density
1          0    24    0.274469  0.000994
2          1    22    0.080087  0.000347
4          2    21    1.448115  0.006896
5          3    17    0.443530  0.003261
3          8    16    0.218503  0.001821
0          4    14    0.435412  0.004785
7          5    12    0.334627  0.005070
8          6    10    0.010824  0.000241
6          7     5    0.006688  0.000669
Co-target layer community stats:
   community  size  w_internal   density
1          0    24   31.454790  0.113967
2          1    22   29.767007  0.128862
4          2    21   43.544542  0.207355
5          3    17   55.467270  0.407848
3          8    16   20.135508  0.167796
0          4    14   11.870341  0.130443
7          5    12   18.091052  0.274107
8          6    10    4.269564  0.094879
6          7     5    3.071080  0.307108


## Gene set enrichment analysis

In [25]:
# 1) Build per-cluster gene sets (TFs + targets)
cluster2genes = build_cluster_gene_sets(
    tf_names=tf_names,
    consensus_labels=consensus_labels,
    current_grn=current_grn,
)

# 2) Load GO BP gene sets (Homo sapiens)
raw_go_bp = gp.get_library(name='GO_Biological_Process_2025', organism='Human')

# 2.1) Extract GO IDs present in this library
go_ids_in_library = extract_go_ids_from_terms(raw_go_bp)

# 2.2) Compute levels only for these GO IDs (BP namespace)
# Note: go_obo_path is optional - defaults to gCRL/data/reference/ontologies/go-basic.obo
go_term_levels = compute_go_levels(
    go_ids=go_ids_in_library,
    namespace="biological_process",
)

Loading GO DAG from /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCAL/gCRL/data/reference/ontologies/go-basic.obo ...
/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCAL/gCRL/data/reference/ontologies/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms


In [26]:

# 3) Filter GO BP by size and (optionally) level
go_bp_filtered = filter_gene_sets_by_size_and_level(
    gene_sets=raw_go_bp,
    min_size=20,               # min genes per GO term
    max_size=200,              # max genes per GO term
    go_term_levels=go_term_levels,
    min_level=5,               # example: only level >= 4
    max_level=None,            # or e.g. 5 if you want 4–5 only
)
len(go_bp_filtered)

1484

In [27]:
# 4) Run ORA with universe = GRN genes (switch to False for library-wide universe)
ora_results = run_ora_for_clusters(
    cluster2genes=cluster2genes,
    gene_sets=go_bp_filtered,
    use_grn_universe=True,     # True: TF+targets universe; False: GO-library universe
    min_genes_in_cluster=5,
)

# 5) Save results
#ora_results.to_csv("tf_cluster_GO_BP_ORA_results.csv", index=False)
#print("Saved ORA results to tf_cluster_GO_BP_ORA_results.csv")

In [29]:
# For each cluster, show top 5 enriched GO BP terms
for cluster_id in ora_results['cluster_id'].unique():
    print(f"\nCluster {cluster_id} top GO BP terms:")
    top_terms = ora_results[ora_results['cluster_id'] == cluster_id].sort_values(by='pval').head(10)
    for _, row in top_terms.iterrows():
        print(f"  {row['go_term']} (P-value: {row['pval']:.4e}), adj p-value: {row['pval_adj']:.4e})")


Cluster 0 top GO BP terms:
  Regulation of Phosphorylation (GO:0042325) (P-value: 2.0400e-03), adj p-value: 1.0000e+00)
  MAPK Cascade (GO:0000165) (P-value: 7.0028e-03), adj p-value: 1.0000e+00)
  + Reg of Transmembrane Receptor Prot Serine/Threonine Kinase Sgnlng Pway (GO:0090100) (P-value: 8.1577e-03), adj p-value: 1.0000e+00)
  Positive Regulation of Autophagy (GO:0010508) (P-value: 1.6280e-02), adj p-value: 1.0000e+00)
  Platelet Aggregation (GO:0070527) (P-value: 1.6280e-02), adj p-value: 1.0000e+00)
  Regulation of DNA Metabolic Process (GO:0051052) (P-value: 3.2446e-02), adj p-value: 1.0000e+00)
  Regulation of Cell Cycle G1/S Phase Transition (GO:1902806) (P-value: 3.2446e-02), adj p-value: 1.0000e+00)
  Protein Maturation (GO:0051604) (P-value: 3.2446e-02), adj p-value: 1.0000e+00)
  Positive Regulation of Cell Growth (GO:0030307) (P-value: 3.2446e-02), adj p-value: 1.0000e+00)
  Positive Regulation of Mitotic Cell Cycle Phase Transition (GO:1901992) (P-value: 3.2446e-02), a

Interpretation of the new 9-cluster solution

As before, I interpret each cluster as a TF-centric regulatory module, using GO BP as a proxy for its dominant program. Adjusted p-values are not significant, so this is a relative interpretation based on ranking.

Cluster 0 – Pro-proliferative kinase / MAPK–TGFβ module

Key terms:

Regulation of phosphorylation; MAPK cascade

Regulation of serine/threonine kinase receptor signaling (TGFβ-like)

Positive regulation of autophagy

Regulation of DNA metabolic process; G1/S transition

Positive regulation of cell growth and mitotic phase transitions

Interpretation:
Cluster 0 looks like an upstream growth and survival signaling module, integrating:

kinase/phosphorylation control (MAPK, TGFβ-like receptors)

cell cycle entry (G1/S)

mild autophagy and protein maturation

Functionally: a pro-proliferative, kinase-driven module that couples growth signaling with DNA metabolism and autophagy as a supporting process.

Cluster 1 – TNF / autophagy / adhesion and translation control

Key terms:

TGFβ-receptor–like pathway regulation; positive regulation of autophagy

Positive regulation of mononuclear cell migration

Regulation of DNA metabolic process

Regulation of translational initiation

Regulation of focal adhesion assembly

Response to TNF

Regulation of cytochrome-c release from mitochondria

Positive regulation of catabolic processes

Interpretation:
This cluster mixes stress and inflammatory cues (TNF) with:

autophagy

control of translational initiation

focal adhesion and complex assembly

Functionally: a stress-integration module that connects TNF/TGFβ-like signaling to autophagy, translation reprogramming, and adhesion remodeling, at the interface between survival and pro-apoptotic signaling (cytochrome-c release).

Cluster 2 – Wnt attenuation and kinase dampening module

Key terms:

Negative regulation of Wnt and canonical Wnt signaling

Regulation of phosphorylation; negative regulation of protein kinase activity

TGFβ-like receptor pathway regulation

Glycerophospholipid biosynthesis

Myeloid cell development

Protein localization to nucleus

Neuron projection development

Positive regulation of autophagy

Interpretation:
Here the leading feature is negative regulation of Wnt and protein kinases, combined with differentiation-related terms (myeloid development, neuron projection).

Functionally: a signal-attenuation / differentiation-gating module, dampening Wnt and kinase activity while allowing differentiation and some autophagy. It looks like a brake on proliferative and developmental signaling.

Cluster 3 – Apoptosis–JAK–STAT and anabolic translation module

Key terms:

Regulation (and positive regulation) of intrinsic apoptotic signaling

Regulation of receptor signaling via JAK–STAT

Positive regulation of autophagy

Platelet aggregation

Regulation of fat cell differentiation

Actin cytoskeleton organization

Cytoplasmic translation; ribosome biogenesis

G1/S transition

Interpretation:
This cluster combines:

intrinsic apoptosis and JAK–STAT signaling

autophagy

ribosome biogenesis and cytoplasmic translation

Functionally: a fate-decision module balancing apoptotic signaling vs anabolic capacity (ribosomes, translation). It looks like a layer where stress/inflammatory JAK–STAT signaling decides between survival and death, while still supporting high biosynthetic potential.

Cluster 4 – Proliferation and structural remodeling module

Key terms:

Positive regulation of cellular component organization

Positive regulation of autophagy

Regulation of fat cell differentiation

G1/S transition; regulation of mitotic transitions

Regulation of phosphorylation; MAPK cascade

Negative regulation of smooth muscle proliferation

Ribosome biogenesis

Interpretation:
Cluster 4 resembles a more “anabolic” sibling of cluster 3:

pro-growth cell cycle terms (G1/S, mitotic transitions)

MAPK + phosphorylation control

autophagy and ribosome biogenesis

structural organization and differentiation

Functionally: a remodeling and proliferation module, linking MAPK signaling to cell cycle progression, organelle/structure organization, and adipogenic or smooth-muscle differentiation constraints.

Cluster 5 – Core cell-cycle and receptor-signaling coordination

Key terms:

Regulation of cell cycle process and mitotic cycle transitions

Regulation of phosphorylation

TGFβ-like receptor signaling regulation

Regulation of JAK–STAT signaling pathway

Regulation of protein localization to plasma membrane / membrane

Regulation of stress fiber assembly

Positive regulation of catabolic processes

Interpretation:
This cluster focuses on:

core mitotic cell cycle control

receptor signaling tuning (TGFβ and JAK–STAT)

membrane localization and cytoskeletal stress fibers

Functionally: a central regulatory hub for cell-cycle execution under receptor-mediated cues, with additional control over receptor localization and actin-based contractility.

Cluster 6 – Immune-regulatory, anti-inflammatory, and kinase-braking module

Key terms:

Negative regulation of phosphate metabolism and phosphorylation

Positive regulation of IL-10 (anti-inflammatory) production

Positive regulation of cell–substrate junction organization

Regulation of erythrocyte differentiation

Negative regulation of type I interferon production

Regulation of mitotic metaphase/anaphase transition

Regulation of stress-activated MAPK cascade

Regulation of mRNA catabolic process

Cellular response to LPS

Interpretation:
This is the clearest immune-regulatory module:

LPS response

anti-inflammatory IL-10 upregulation

suppression of type I interferon

brake on phosphorylation and stress-activated MAPK

regulation of mRNA decay

Functionally: a feedback/attenuation module that damps inflammatory signaling (MAPK, IFN) while promoting IL-10, and tunes mRNA turnover and junctions. This is an immune homeostasis / resolution module.

Cluster 7 – Pro-proliferative and motility program (myeloid / EMT-flavoured)

Key terms:

Mitotic phase transitions; G1/S transition; positive regulation of these

Positive regulation of stress fiber assembly

Myeloid cell development

Regulation of TGFβ receptor signaling

Positive regulation of biosynthetic processes

Regulation of neuron apoptosis

Positive regulation of cell spreading and substrate-dependent adhesion

Positive regulation of autophagy

Interpretation:
Cluster 7 combines:

strong cell cycle and biosynthetic drive

myeloid development

motility/adhesion (stress fibers, spreading, adhesion-dependent processes)

TGFβ signaling regulation

Functionally: a proliferative and motile program, especially compatible with myeloid cells or cells undergoing stress fiber–dependent migration, with TGFβ signaling as a modulator.

Cluster 8 – Hypoxia–EMT and morphogenetic remodeling module

Key terms:

Cellular response to decreased oxygen (hypoxia)

Regulation (and positive regulation) of EMT

Protein localization to nucleus

Cell projection organization

Regulation of fat cell differentiation

Regulation of cell cycle process

Regulation of actin cytoskeleton organization

Regulation of epithelial cell proliferation

Response to UV

Interpretation:
This cluster is clearly focused on:

hypoxia response

EMT regulation

cytoskeletal and projection remodeling

epithelial proliferation and UV/stress responses

Functionally: a hypoxia- and stress-driven EMT/morphogenesis module, coordinating cytoskeletal changes, protrusions, and fate decisions (epithelial vs mesenchymal, adipogenic contributions).

Comparison with the previous 5-cluster solution
1. Conceptual mapping between the two solutions

Rough correspondences:

Old Cluster 1 (MAPK + immune receptor signaling)
→ Now spread mainly across Cluster 6 (immune regulation, LPS, IL-10, IFN) and parts of Cluster 7 (myeloid development, motility) and Cluster 0/5 (MAPK + cell cycle).

Old Cluster 2 (TNF response, growth suppression, autophagy)
→ Now decomposed into Cluster 1 (TNF, autophagy, translation/adhesion) and Cluster 2 (Wnt attenuation, kinase dampening, autophagy).

Old Cluster 3 (ECM, apoptosis, EMT, cytoskeleton)
→ Now mostly in Cluster 3 (intrinsic apoptosis + JAK–STAT), Cluster 8 (hypoxia + EMT + cytoskeleton), and parts of Cluster 4/7 (motility, differentiation).

Old Cluster 4 (phosphorylation, hypoxia, TGFβ, cell cycle)
→ Now more finely split between Cluster 0, 2, 4, 5, 7, 8, all with various combinations of phosphorylation control, TGFβ-like signaling, cell cycle, and stress.

Old Cluster 0 (cytoskeleton, metabolic adaptation, hypoxia, TGFβ)
→ Now mostly recapitulated in Cluster 8 (hypoxia + EMT + actin) and Cluster 4/7 (organization + proliferation).

In short, the 9-cluster solution refines the old functional modules, teasing apart:

immune activation vs immune resolution (old 1/2 → new 6 vs 1/2/7),

Wnt-specific attenuation (new 2),

clearer hypoxia–EMT vs apoptosis–JAK–STAT (new 8 vs 3).

2. Overlap of GO BP terms and functional disentanglement

If we look at GO BP overlap across clusters in the new solution, several very generic regulatory terms appear in many clusters:

Regulation of phosphorylation (or its negative form)
present in clusters 0, 2, 4, 5, 6.

Positive regulation of autophagy
present in clusters 0, 1, 2, 3, 4, 7.

Cell cycle phase transitions (G1/S, mitotic)
widely shared (0, 3, 4, 5, 7).

MAPK-related terms
in clusters 0, 4, 6 (stress-activated), and indirectly via TNF and TGFβ.

Fat cell differentiation
appears in 3, 4, 8.

Ribosome biogenesis
in 3 and 4.

Myeloid cell development
in 2 and 7.

Several EMT / cytoskeleton / adhesion-related terms are shared between 3, 4, 7, 8.

In the previous 5-cluster solution, there was also overlap (e.g., MAPK and TGFβ terms in multiple clusters, shared autophagy and phosphorylation), but:

Each cluster had a more distinct “headline” theme:

Old 1: receptor/MAPK immune activation

Old 2: TNF + growth arrest + autophagy

Old 3: ECM + apoptosis + EMT

Old 4: phosphorylation + hypoxia + TGFβ

Old 0: cytoskeleton + metabolic adaptation

In contrast, in the new 9-cluster solution:

The same high-level regulatory GO terms recur across many clusters (phosphorylation, cell cycle, autophagy), and

Several clusters share not only generic but also semi-specific terms (e.g. autophagy present in 0,1,2,3,4,7; G1/S in 0,3,4,7; fat cell differentiation in 3,4,8; Wnt/kinase attenuation in 2 and indirectly in 6).

This suggests that functional GO BP profiles are more entangled across the 9 clusters.

3. Which solution is more functionally disentangled?

If we define functional disentanglement as:

Clear, non-overlapping biological themes per cluster

Minimal reuse of the same GO BP terms across many clusters

then, based on these lists:

The 5-cluster solution had fewer modules with broader but relatively distinct themes (immune activation, stress/autophagy growth arrest, ECM/EMT, kinase/hypoxia, cytoskeletal/metabolic adaptation).

The 9-cluster solution refines those into sub-modules (e.g., Wnt attenuation vs JAK–STAT vs immune resolution vs hypoxia–EMT), but at the cost of substantial overlap of generic GO categories across many clusters.

Conclusion:

In terms of GO BP functional disentanglement, the original 5-cluster solution appears slightly more disentangled, because each cluster had a more unique, higher-level biological “role” with less repeated annotation.

The 9-cluster solution likely better reflects topological or expression-based substructure of the TF–target network, but it is more entangled at the level of broad GO BP terms, especially for shared processes like phosphorylation, autophagy, and cell cycle.

